# CNN Stock Predictor — demo notebook

All logic now lives in [`src/`](src/). This notebook is just a thin demo.

Modules:
- [`src/data.py`](src/data.py) — CSV / yfinance loaders, train/val split, scaling
- [`src/models.py`](src/models.py) — model factories + hyperparameter ranges
- [`src/training.py`](src/training.py) — unified `train()` entry point
- [`src/evaluation.py`](src/evaluation.py) — `predict_and_evaluate()` returning predictions + MAE/MAPE
- [`src/hyperparam_search.py`](src/hyperparam_search.py) — (1+1)-ES over CNN hyperparameters
- [`src/plotting.py`](src/plotting.py) — training-curve / prediction plots

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src import (
    discover_csv_paths,
    load_yfinance,
    prepare_dataset,
    train,
    train_on_ticker,
    predict_and_evaluate,
    one_plus_one_es,
    plot_predictions,
)

sns.set_style('whitegrid')
plt.style.use('ggplot')

## 1. Discover local data

Looks for files matching `stock_market_data/sp500/csv/*.csv`.

In [ ]:
csv_paths = discover_csv_paths()
print(f'Found {len(csv_paths)} local CSVs')
list(csv_paths)[:5]

## 2. Train the best-known CNN on IBM

`train_on_ticker` returns a `TrainingResult` with the model, the prepared `Dataset` (incl. the fitted scaler), and the full Keras history.

In [ ]:
result = train_on_ticker('IBM', csv_paths, plot=True)
print(f'final val MAPE = {result.final_val_mape:.3f}')
print(f'final val MAE  = {result.final_val_mae:.3f}')

## 3. Evaluate on unseen data (MSFT, AAPL, PLTR)

`predict_and_evaluate` returns predictions, actuals, and metrics — unlike the previous version which silently discarded them.

In [ ]:
microsoft_df = pd.DataFrame({
    'Open':   [383.76, 371.01, 332.38, 315.13, 320.26, 347.11, 334.36, 309.83, 287.23, 274.88, 368.23],
    'High':   [384.30, 371.95, 333.83, 315.88, 322.41, 351.89, 337.96, 313.71, 290.45, 275.00, 371.45],
    'Low':    [377.44, 367.35, 326.36, 310.02, 319.21, 345.07, 333.45, 309.83, 285.67, 269.52, 366.32],
    'Close':  [378.85, 370.27, 327.73, 312.14, 321.01, 350.98, 335.02, 311.74, 287.18, 273.78, 370.95],
    'Volume': [28942500, 27683900, 21072400, 26297600, 24342600, 41637700, 23084700, 26730300, 25824300, 34558700, 23099800],
})
msft_next_close = [380, 369.67, 328.65, 312.79, 324.04, 337.77, 328.60, 314.00, 284.34, 272.29, 372]

msft_eval = predict_and_evaluate(result.model, result.dataset.scaler, microsoft_df, msft_next_close)
print(f'MSFT MAE = {msft_eval.mae:.3f}, MAPE = {msft_eval.mape:.2f}%')
msft_eval.as_dataframe()

In [ ]:
apple_df = pd.DataFrame({
    'Open':   [130.92, 190.87, 134.83, 152.35, 153.11, 134.08, 151.28, 165.80, 170.98, 196.02, 193.63, 194.20, 193.11],
    'High':   [132.42, 190.90, 137.29, 153.00, 155.50, 136.25, 153.40, 168.16, 174.30, 197.20, 195.00, 195.99, 193.49],
    'Low':    [129.64, 189.25, 134.13, 150.85, 152.88, 133.77, 150.10, 165.54, 170.76, 192.55, 193.59, 193.67, 191.42],
    'Close':  [131.86, 189.97, 135.94, 152.55, 155.33, 135.27, 152.59, 167.63, 173.57, 193.22, 194.27, 195.71, 193.18],
    'Volume': [63814900, 24048300, 63646600, 59144100, 65573800, 58280400, 73695900, 47720200, 113316400, 47460200, 47477700, 53377300, 60896700],
})
aapl_next_close = [129.31, 189.79, 135.21, 148.48, 153.71, 137.87, 152.99, 166.65, 173.50, 195.32, 195.71, 193.12, 193.4]

aapl_eval = predict_and_evaluate(result.model, result.dataset.scaler, apple_df, aapl_next_close)
print(f'AAPL MAE = {aapl_eval.mae:.3f}, MAPE = {aapl_eval.mape:.2f}%')
plot_predictions(aapl_eval.actual, aapl_eval.predictions, title='AAPL — next-day close')

## 4. Train on a live-fetched ticker via yfinance

In [ ]:
apps_df = load_yfinance('APPS')
apps_result = train(apps_df, epochs=300, plot=False)
print(f'APPS final val MAPE = {apps_result.final_val_mape:.3f}')

## 5. Hyperparameter evolution — (1+1)-ES

Evolves the CNN configuration (filter count, kernel size, activations, optimizer, loss). Each iteration trains a fresh network from scratch, so this is slow — defaults below run ~20 iterations with 100 epochs and early stopping. Raise `max_iterations` for serious search.

In [ ]:
dataset = prepare_dataset(apps_df)  # prepare once, reused across all fitness evaluations
history = one_plus_one_es(
    dataset,
    max_iterations=20,
    reset_threshold=10,
    mutation_probability=0.3,
    epochs=100,
    early_stopping_patience=10,
    seed=42,
)
print('best fitness:', history.best_fitness)
print('best params: ', history.best_params)
print(f'evaluations: {history.evaluations}, cache hits: {history.cache_hits}')